# Melatih pembaca layar alat SmartTriage AI (YOLO digit 0-9)

Notebook ini dijalankan di Google Colab dengan GPU (Runtime > Change runtime type > T4 GPU).
Dataset diambil dari Roboflow memakai akun Anda sendiri. API key **tidak** ditulis di notebook,
tetapi disimpan di Colab Secrets dengan nama `ROBOFLOW_API_KEY` (ikon kunci di panel kiri).

Model yang dihasilkan mendeteksi digit 0-9. Nama kelas dataset boleh berbeda (misalnya `seven` atau
`digit_7`); skrip memetakannya ke 0-9 dan membuang kelas lain seperti `HR` atau `SpO2`.

In [ ]:
!git clone https://github.com/imamabidin69-svg/SmartTriageAI.git
%cd SmartTriageAI/backend
!pip install -q -r requirements.txt -r ai_training/requirements-training.txt

## 1. Pilih dataset

Isi workspace, nama proyek, dan versi sesuai halaman dataset di Roboflow Universe. Klik **Fork Project**
dulu supaya dataset masuk ke workspace Anda. Kandidat yang bisa dicek:

- https://universe.roboflow.com/midprep/cloudphysician_part_1
- https://universe.roboflow.com/abu-hammad-dhylo/cardiac-monitor
- https://universe.roboflow.com/amogh-bhagwat-ywg29/icu-metrics
- https://universe.roboflow.com/datacluster-labs-agryi/bp-monitor-reading-medical-device-images

Pastikan dataset punya anotasi per digit (kelas 0 sampai 9). Bila yang dianotasi hanya kotak parameter
(HR, SpO2, dan seterusnya), tambahkan anotasi digit di Roboflow sebelum diekspor.

In [ ]:
import os
from pathlib import Path

from google.colab import userdata

from ai_training.dataset import unduh_dari_roboflow

os.environ["ROBOFLOW_API_KEY"] = userdata.get("ROBOFLOW_API_KEY")

WORKSPACE = "isi-workspace-anda"
PROYEK = "isi-nama-proyek"
VERSI = 1

data_yaml = unduh_dari_roboflow(WORKSPACE, PROYEK, VERSI, Path("/content/dataset"))
print(data_yaml.read_text())

## 2. Latih dan ekspor ke ONNX

In [ ]:
!python -m ai_training.latih --data {data_yaml} --epoch 100
!python -m ai_training.ekspor --bobot runs/detect/digit_layar/weights/best.pt

## 3. Evaluasi (target 95% angka terbaca benar)

Unggah foto uji ke `/content/uji` beserta `label.csv` (lihat `ai_training/README.md`).

In [ ]:
MODEL = "runs/detect/digit_layar/weights/best.onnx"
!python -m ai_training.evaluasi --model {MODEL} --foto /content/uji --label /content/uji/label.csv

## 4. Unduh model

Salin `best.onnx` ke `backend/app/ai/artefak/yolo_digit.onnx` di laptop, lalu jalankan ulang backend.
`GET /api/v1/health` akan menampilkan `pembaca_layar: ok`.

In [ ]:
from google.colab import files

files.download("runs/detect/digit_layar/weights/best.onnx")